# Horvitz-Thompson Exposure Contrasts

## Estimating the contrast between exposure means

For exposure $d\in\{0,1\}$, let $Y_i(d)$ be unit $i$'s potential outcome and let $\pi_i(d)=\Pr\{D_i(Z)=d\}$ be its known marginal exposure probability. The Horvitz-Thompson mean is

$$
\widehat\mu(d)=\frac{1}{N}\sum_{i=1}^{N}\frac{\mathbf 1\{D_i=d\}Y_i}{\pi_i(d)}.
$$

Here, $N$ is the number of units, $D_i$ is the realized exposure, and $Y_i=Y_i(D_i)$ is the observed outcome. The mean exposure contrast is

$$
\widehat\tau_{1,0}=\widehat\mu(1)-\widehat\mu(0).
$$

Known positive probabilities make the two weighted means unbiased under the declared randomization design. A variance calculation would also require joint exposure probabilities, which are not supplied to the released calculation.

In [1]:
from pathlib import Path
import io
import itertools
import math
import sys
import zipfile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT
import RiskLabAI
from RiskLabAI.causal_factor_analysis import exposure_mean_contrast
assert RiskLabAI.__version__ == '3.2.0'
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '07_network_interference'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '07_network_interference'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '07_network_interference'
DATA_FILE = ensure_dataset(BOOK_ROOT, 'french_49_industries')
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return '0'
        if value.is_integer():
            return f'{int(value):,}'
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f'{value:.6g}'
        return f'{value:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    latex = latex.replace('\\begin{table}[htbp]', '\\begin{table}[htbp]\n\\centering', 1)
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)
    plt.close(figure)

def read_public_returns():
    with zipfile.ZipFile(DATA_FILE) as archive:
        lines = archive.read(archive.namelist()[0]).decode('latin1').splitlines()
    header = next((index for index, line in enumerate(lines) if line.startswith(',Agric')))
    ending = next((index for index in range(header + 1, len(lines)) if not lines[index].strip()))
    raw = pd.read_csv(io.StringIO('\n'.join(lines[header:ending])))
    raw = raw.rename(columns={raw.columns[0]: 'Date'})
    raw['Date'] = pd.to_datetime(raw['Date'].astype(str), format='%Y%m%d')
    raw = raw.set_index('Date').replace([-99.99, -999.0], np.nan) / 100
    return raw.loc['2019-01-01':'2023-12-31'].dropna().copy()

def correlation_graph(correlation, neighbors):
    values = correlation.to_numpy()
    size = len(correlation)
    adjacency = np.zeros((size, size), dtype=int)
    for row in range(size):
        order = np.argsort(values[row])
        selected = [column for column in order[::-1] if column != row][:neighbors]
        adjacency[row, selected] = 1
    return np.maximum(adjacency, adjacency.T)

def line_graph(size):
    adjacency = np.zeros((size, size), dtype=int)
    for index in range(size - 1):
        adjacency[index, index + 1] = adjacency[index + 1, index] = 1
    return adjacency

def ring_graph(size, radius=1):
    adjacency = np.zeros((size, size), dtype=int)
    for index in range(size):
        for step in range(1, radius + 1):
            adjacency[index, (index + step) % size] = 1
            adjacency[index, (index - step) % size] = 1
    return adjacency

def star_graph(size):
    adjacency = np.zeros((size, size), dtype=int)
    adjacency[0, 1:] = 1
    adjacency[1:, 0] = 1
    return adjacency

def complete_graph(size):
    return np.ones((size, size), dtype=int) - np.eye(size, dtype=int)

def exposure_labels(adjacency, assignment, threshold=1):
    return (adjacency @ np.asarray(assignment) >= threshold).astype(int)

def exposure_probabilities(adjacency, assignment_probability, threshold=1):
    degrees = adjacency.sum(axis=1).astype(int)
    positive = np.array([1 - sum((math.comb(int(degree), count) * assignment_probability ** count * (1 - assignment_probability) ** (degree - count) for count in range(min(threshold, int(degree) + 1)))) for degree in degrees])
    return np.column_stack([1 - positive, positive])

def assignment_probability(assignment, probability):
    treated = int(np.asarray(assignment).sum())
    return probability ** treated * (1 - probability) ** (len(assignment) - treated)

def circle_positions(size):
    angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, size, endpoint=False)
    return np.column_stack([np.cos(angles), np.sin(angles)])

def draw_network(axis, adjacency, labels, node_values=None, title=''):
    positions = circle_positions(len(labels))
    for first in range(len(labels)):
        for second in range(first + 1, len(labels)):
            if adjacency[first, second]:
                axis.plot(positions[[first, second], 0], positions[[first, second], 1], color=PALETTE['grid'], linewidth=1.5, alpha=0.8, zorder=1)
    values = adjacency.sum(axis=1) if node_values is None else np.asarray(node_values)
    colors = [PALETTE['cyan'] if value <= np.median(values) else PALETTE['amber'] for value in values]
    axis.scatter(positions[:, 0], positions[:, 1], s=420, c=colors, edgecolors=PALETTE['text'], linewidths=1.0, zorder=2)
    for index, label in enumerate(labels):
        axis.text(positions[index, 0], positions[index, 1], str(label), ha='center', va='center', color=PALETTE['background'], fontsize=8.5, fontweight='bold', zorder=3)
    axis.set_title(title)
    axis.set_aspect('equal')
    axis.axis('off')
    return positions

def capture_failure(name, calculation):
    try:
        calculation()
        return {'Check': name, 'Rejected': False, 'Explanation': 'Calculation continued'}
    except ValueError:
        return {'Check': name, 'Rejected': True, 'Explanation': 'Stopped as required'}
assignments = list(itertools.product([0, 1], repeat=3))
ring_three = ring_graph(3)
potential = np.array([[1, 2], [3, 7], [2, 1]], dtype=float)
probabilities = np.array([[0.25, 0.75]] * 3)
exact_rows = []
for assignment in assignments:
    exposure = exposure_labels(ring_three, assignment)
    outcome = potential[np.arange(3), exposure]
    result = exposure_mean_contrast(outcome, exposure, probabilities, first=1, second=0)
    exact_rows.append({'Assignment': ''.join(map(str, assignment)), 'Exposure': ''.join(map(str, exposure)), 'First mean': result['first_mean'], 'Second mean': result['second_mean'], 'Contrast': result['contrast']})
exact_assignments = pd.DataFrame(exact_rows)
truth = float(np.mean(potential[:, 1] - potential[:, 0]))
exact_average = float(exact_assignments['Contrast'].mean())
assert np.isclose(exact_average, truth)
potential_table = pd.DataFrame({'Unit': np.arange(3), 'Outcome with no exposed neighbor': potential[:, 0], 'Outcome with an exposed neighbor': potential[:, 1], 'Exposure effect': potential[:, 1] - potential[:, 0]})
save_table(potential_table, 'ht_exact_potential_outcomes', 'Potential outcomes in the released three-unit exact design.', 'tab:cim7-ht-exact-potential-outcomes', fit=True)
save_table(exact_assignments, 'ht_exact_assignments', 'All assignments in the released three-unit exact design.', 'tab:cim7-ht-exact-assignments', fit=True)
exact_summary = pd.DataFrame([{'Quantity': 'Assignments', 'Value': len(assignments)}, {'Quantity': 'Known average exposure effect', 'Value': truth}, {'Quantity': 'Average released contrast', 'Value': exact_average}, {'Quantity': 'Absolute difference', 'Value': abs(exact_average - truth)}])
save_table(exact_summary, 'ht_exact_summary', 'Exact randomization expectation for the released exposure contrast.', 'tab:cim7-ht-exact-summary')
example_assignment = np.array(assignments[5])
example_exposure = exposure_labels(ring_three, example_assignment)
example_outcome = potential[np.arange(3), example_exposure]
example_result = exposure_mean_contrast(example_outcome, example_exposure, probabilities, first=1, second=0)
contribution_table = pd.DataFrame({'Unit': np.arange(3), 'Exposure': example_exposure, 'Observed outcome': example_outcome, 'Unit contribution': example_result['unit_contributions']})
save_table(contribution_table, 'ht_exact_contributions', 'Unit contributions for one exact-design assignment.', 'tab:cim7-ht-exact-contributions')
returned_fields = pd.DataFrame([{'Returned quantity': 'First exposure mean', 'Available': 'Yes'}, {'Returned quantity': 'Second exposure mean', 'Available': 'Yes'}, {'Returned quantity': 'Mean contrast', 'Available': 'Yes'}, {'Returned quantity': 'Unit contributions', 'Available': 'Yes'}, {'Returned quantity': 'Variance', 'Available': 'No'}, {'Returned quantity': 'Confidence interval', 'Available': 'No'}])
save_table(returned_fields, 'ht_returned_fields', 'Quantities returned by the released exposure calculation.', 'tab:cim7-ht-returned-fields')
exact_assignments.to_csv(FIGURE_DATA / 'ht_exact_assignments.csv', index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.bar(exact_assignments['Assignment'], exact_assignments['Contrast'], color=PALETTE['cyan'])
axis.axhline(truth, color=PALETTE['amber'], linewidth=2, linestyle='--', label='Known average effect')
axis.set_xlabel('Assignment vector')
axis.set_ylabel('Estimated contrast')
axis.set_title('Released Contrast Across All Three-Unit Assignments')
axis.legend()
finish_axis(axis)
save_figure(figure, 'ht_exact_assignments')


,Unit,Outcome with no exposed neighbor,Outcome with an exposed neighbor,Exposure effect
0,0,1,2,1
1,1,3,7,4
2,2,2,1,-1


,Assignment,Exposure,First mean,Second mean,Contrast
0,000,000,0,8,-8
1,001,110,4,2.666667,1.333333
2,010,101,1.333333,4,-2.666667
3,011,111,4.444444,0,4.444444
4,100,011,3.555556,1.333333,2.222222
5,101,111,4.444444,0,4.444444
6,110,111,4.444444,0,4.444444
7,111,111,4.444444,0,4.444444


,Quantity,Value
0,Assignments,8
1,Known average exposure effect,1.333333
2,Average released contrast,1.333333
3,Absolute difference,2.22045e-16


,Unit,Exposure,Observed outcome,Unit contribution
0,0,1,2,2.666667
1,1,1,7,9.333333
2,2,1,1,1.333333


,Returned quantity,Available
0,First exposure mean,Yes
1,Second exposure mean,Yes
2,Mean contrast,Yes
3,Unit contributions,Yes
4,Variance,No
5,Confidence interval,No


<Figure size 880x520 with 1 Axes>

## Exact Enumeration Across Four Graphs

For a graph $G$ and independent assignment probability $p$, let $\mathcal Z$ contain all $2^N$ assignments. The design expectation is

$$
\mathbb E_Z[\widehat\tau_{1,0}]
=\sum_{z\in\mathcal Z}\Pr(Z=z)\widehat\tau_{1,0}(z).
$$

The factor $\Pr(Z=z)$ is determined before outcomes are observed. Exact enumeration checks the mean contrast for line, ring, star, and complete graphs without relying on a large-sample approximation.

In [2]:
graph_builders = {
    "Line": line_graph(4),
    "Ring": ring_graph(4),
    "Star": star_graph(4),
    "Complete": complete_graph(4),
}
enumeration_probability = 0.35
graph_rows = []
graph_estimate_rows = []
for graph_name, graph in graph_builders.items():
    graph_degrees = graph.sum(axis=1)
    graph_probabilities = exposure_probabilities(graph, enumeration_probability)
    baseline = 1 + 0.2 * np.arange(4) + 0.3 * graph_degrees
    effects = 0.5 + 0.1 * graph_degrees
    expected_estimate = 0
    estimate_minimum = np.inf
    estimate_maximum = -np.inf
    for assignment in itertools.product([0, 1], repeat=4):
        exposure = exposure_labels(graph, assignment)
        outcome = baseline + effects * exposure
        estimate = exposure_mean_contrast(outcome, exposure, graph_probabilities, first=1, second=0)["contrast"]
        probability = assignment_probability(assignment, enumeration_probability)
        expected_estimate += probability * estimate
        estimate_minimum = min(estimate_minimum, estimate)
        estimate_maximum = max(estimate_maximum, estimate)
        graph_estimate_rows.append({"Graph": graph_name, "Assignment": "".join(map(str, assignment)), "Assignment probability": probability, "Estimate": estimate})
    graph_truth = float(effects.mean())
    graph_rows.append({
        "Graph": graph_name,
        "Edges": int(graph.sum() / 2),
        "Minimum degree": int(graph_degrees.min()),
        "Maximum degree": int(graph_degrees.max()),
        "Known average effect": graph_truth,
        "Exact expected estimate": expected_estimate,
        "Absolute error": abs(expected_estimate - graph_truth),
        "Minimum assignment estimate": estimate_minimum,
        "Maximum assignment estimate": estimate_maximum,
    })
graph_enumeration = pd.DataFrame(graph_rows)
graph_estimates = pd.DataFrame(graph_estimate_rows)
assert np.allclose(graph_enumeration["Known average effect"], graph_enumeration["Exact expected estimate"])
save_table(graph_enumeration, "ht_graph_enumeration", "Exact exposure-contrast enumeration across four small graphs.", "tab:cim7-ht-graph-enumeration", fit=True)

graph_enumeration.to_csv(FIGURE_DATA / "ht_graph_expectations.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
positions_graph = np.arange(len(graph_enumeration))
axis.bar(positions_graph - 0.18, graph_enumeration["Known average effect"], 0.36, color=PALETTE["amber"], label="Known effect")
axis.bar(positions_graph + 0.18, graph_enumeration["Exact expected estimate"], 0.36, color=PALETTE["cyan"], label="Exact estimator mean")
axis.set_xticks(positions_graph, graph_enumeration["Graph"])
axis.set_ylabel("Exposure contrast")
axis.set_title("Exact Design Expectations Across Four Graphs")
axis.legend()
finish_axis(axis)
save_figure(figure, "ht_graph_expectations")

,Graph,Edges,Minimum degree,Maximum degree,Known average effect,Exact expected estimate,Absolute error,Minimum assignment estimate,Maximum assignment estimate
0,Line,3,1,2,0.65,0.65,2.22045e-16,-3.47929,5.393939
1,Ring,4,2,2,0.7,0.7,5.55112e-16,-4.497041,4.502165
2,Star,3,1,3,0.65,0.65,2.22045e-16,-3.69117,5.859125
3,Complete,6,3,3,0.8,0.8,6.66134e-16,-8.010924,4.135792


<Figure size 880x520 with 1 Axes>

## A Controlled Experiment on the Public Topology

The public graph is now held fixed, but assignments and potential outcomes are generated. Unit $i$ has

$$
Y_i(0)=b_i,\qquad Y_i(1)=b_i+\tau_i,
$$

where $b_i$ is a fixed generated baseline and $\tau_i=0.4+0.1d_i$ is a declared exposure effect based on degree $d_i$. The known average effect is $N^{-1}\sum_i\tau_i$. This design tests the calculation; it does not turn the public correlations into causal evidence.

In [3]:
public_returns = read_public_returns()
selected_returns = public_returns.iloc[:, :12]
industries = selected_returns.columns.to_list()
adjacency = correlation_graph(selected_returns.corr(), 2)
degrees = adjacency.sum(axis=1)
public_probability = 0.25
public_probabilities = exposure_probabilities(adjacency, public_probability)
baseline_rng = np.random.default_rng(7202)
baseline = 0.5 + 0.3 * degrees + baseline_rng.normal(0, 0.2, len(industries))
effects = 0.4 + 0.1 * degrees
known_effect = float(effects.mean())

generated_design = pd.DataFrame({
    "Industry": industries,
    "Degree": degrees,
    "Generated baseline": baseline,
    "Generated exposure effect": effects,
    "Probability of exposure zero": public_probabilities[:, 0],
    "Probability of exposure one": public_probabilities[:, 1],
})
save_table(generated_design, "ht_public_generated_design", "Generated potential-outcome design on the fixed public topology.", "tab:cim7-ht-public-generated-design", fit=True)

exact_rows = []
for assignment in itertools.product([0, 1], repeat=len(industries)):
    exposure = exposure_labels(adjacency, assignment)
    outcome = baseline + effects * exposure
    result = exposure_mean_contrast(outcome, exposure, public_probabilities, first=1, second=0)
    probability = assignment_probability(assignment, public_probability)
    naive = outcome[exposure == 1].mean() - outcome[exposure == 0].mean() if 0 < exposure.sum() < len(exposure) else np.nan
    exact_rows.append({"Assignment probability": probability, "Exposed units": int(exposure.sum()), "Horvitz-Thompson contrast": result["contrast"], "Naive observed difference": naive})
public_exact = pd.DataFrame(exact_rows)
exact_ht_mean = float(np.sum(public_exact["Assignment probability"] * public_exact["Horvitz-Thompson contrast"]))
valid_probability = float(public_exact.loc[public_exact["Naive observed difference"].notna(), "Assignment probability"].sum())
exact_naive_mean = float(np.sum(public_exact["Assignment probability"] * public_exact["Naive observed difference"].fillna(0)) / valid_probability)
assert np.isclose(exact_ht_mean, known_effect)

exact_summary = pd.DataFrame([
    {"Quantity": "Assignments enumerated", "Value": len(public_exact)},
    {"Quantity": "Known average exposure effect", "Value": known_effect},
    {"Quantity": "Exact Horvitz-Thompson expectation", "Value": exact_ht_mean},
    {"Quantity": "Exact expectation error", "Value": exact_ht_mean - known_effect},
    {"Quantity": "Naive expectation when both groups appear", "Value": exact_naive_mean},
    {"Quantity": "Probability both exposure groups appear", "Value": valid_probability},
])
save_table(exact_summary, "ht_public_exact_summary", "Exact design expectations on the fixed public topology.", "tab:cim7-ht-public-exact-summary", fit=True)

simulation_count = 5000
simulation_rng = np.random.default_rng(7203)
simulation_rows = []
for repetition in range(simulation_count):
    assignment = simulation_rng.binomial(1, public_probability, len(industries))
    exposure = exposure_labels(adjacency, assignment)
    outcome = baseline + effects * exposure
    result = exposure_mean_contrast(outcome, exposure, public_probabilities, first=1, second=0)
    naive = outcome[exposure == 1].mean() - outcome[exposure == 0].mean() if 0 < exposure.sum() < len(exposure) else np.nan
    simulation_rows.append({"Repetition": repetition + 1, "Exposed units": int(exposure.sum()), "Horvitz-Thompson contrast": result["contrast"], "Naive observed difference": naive})
simulation = pd.DataFrame(simulation_rows)
simulation["Running Horvitz-Thompson mean"] = simulation["Horvitz-Thompson contrast"].expanding().mean()
simulation["Running naive mean"] = simulation["Naive observed difference"].expanding().mean()

repeated_summary = pd.DataFrame([
    {"Estimator": "Horvitz-Thompson", "Mean": simulation["Horvitz-Thompson contrast"].mean(), "Standard deviation": simulation["Horvitz-Thompson contrast"].std(), "Error from truth": simulation["Horvitz-Thompson contrast"].mean() - known_effect, "Valid repetitions": simulation_count},
    {"Estimator": "Naive observed difference", "Mean": simulation["Naive observed difference"].mean(), "Standard deviation": simulation["Naive observed difference"].std(), "Error from truth": simulation["Naive observed difference"].mean() - known_effect, "Valid repetitions": int(simulation["Naive observed difference"].notna().sum())},
])
save_table(repeated_summary, "ht_public_repeated_summary", "Repeated generated assignments on the fixed public topology.", "tab:cim7-ht-public-repeated-summary", fit=True)

quantile_rows = []
for estimator in ["Horvitz-Thompson contrast", "Naive observed difference"]:
    series = simulation[estimator].dropna()
    quantile_rows.append({"Estimator": estimator, "5th percentile": series.quantile(0.05), "Median": series.median(), "95th percentile": series.quantile(0.95), "Minimum": series.min(), "Maximum": series.max()})
quantiles = pd.DataFrame(quantile_rows)
save_table(quantiles, "ht_public_quantiles", "Finite-sample distributions under repeated generated assignments.", "tab:cim7-ht-public-quantiles", fit=True)

first_assignment = simulation_rng.binomial(1, public_probability, len(industries))
first_exposure = exposure_labels(adjacency, first_assignment)
first_outcome = baseline + effects * first_exposure
first_result = exposure_mean_contrast(first_outcome, first_exposure, public_probabilities, first=1, second=0)
example_contributions = pd.DataFrame({"Industry": industries, "Exposure": first_exposure, "Observed outcome": first_outcome, "Unit contribution": first_result["unit_contributions"]})
save_table(example_contributions, "ht_public_contributions", "Unit contributions under one generated public-topology assignment.", "tab:cim7-ht-public-contributions", fit=True)

invalid_checks = pd.DataFrame([
    capture_failure("Exposure label outside the supplied columns", lambda: exposure_mean_contrast([1], [2], [[0.5, 0.5]], first=0, second=1)),
    capture_failure("Contrasted exposures are identical", lambda: exposure_mean_contrast([1], [0], [[0.5, 0.5]], first=0, second=0)),
    capture_failure("A contrasted exposure has zero probability", lambda: exposure_mean_contrast([1], [0], [[1, 0]], first=0, second=1)),
    capture_failure("Probability row exceeds one", lambda: exposure_mean_contrast([1], [0], [[0.6, 0.6]], first=0, second=1)),
])
assert invalid_checks["Rejected"].all()
save_table(invalid_checks, "ht_invalid_input_checks", "Plain-language checks for invalid exposure calculations.", "tab:cim7-ht-invalid-input-checks", fit=True)

public_exact.to_csv(FIGURE_DATA / "ht_public_exact_distribution.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.scatter(public_exact["Exposed units"] + np.linspace(-0.12, 0.12, len(public_exact)), public_exact["Horvitz-Thompson contrast"], s=10, alpha=0.35, color=PALETTE["cyan"])
axis.axhline(known_effect, color=PALETTE["amber"], linewidth=2, linestyle="--", label="Known average effect")
axis.set_xlabel("Number of exposed units")
axis.set_ylabel("Horvitz-Thompson contrast")
axis.set_title("All Assignments on the Fixed Public Topology")
axis.legend()
finish_axis(axis)
save_figure(figure, "ht_public_exact_distribution")

simulation.to_csv(FIGURE_DATA / "ht_public_repeated_estimates.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.hist(simulation["Horvitz-Thompson contrast"], bins=45, density=True, alpha=0.72, color=PALETTE["cyan"], label="Horvitz-Thompson")
axis.hist(simulation["Naive observed difference"].dropna(), bins=35, density=True, alpha=0.65, color=PALETTE["rose"], label="Naive observed difference")
axis.axvline(known_effect, color=PALETTE["amber"], linewidth=2, linestyle="--", label="Known average effect")
axis.set_xlabel("Estimated exposure contrast")
axis.set_ylabel("Density")
axis.set_title("Repeated Generated Assignments on the Fixed Topology")
axis.legend()
finish_axis(axis)
save_figure(figure, "ht_public_repeated_estimates")

figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.plot(simulation["Repetition"], simulation["Running Horvitz-Thompson mean"], color=PALETTE["cyan"], linewidth=1.5, label="Running Horvitz-Thompson mean")
axis.plot(simulation["Repetition"], simulation["Running naive mean"], color=PALETTE["rose"], linewidth=1.5, label="Running naive mean")
axis.axhline(known_effect, color=PALETTE["amber"], linewidth=2, linestyle="--", label="Known average effect")
axis.set_xlabel("Generated assignments")
axis.set_ylabel("Running mean")
axis.set_title("Randomization Averages Under the Declared Design")
axis.legend()
finish_axis(axis)
save_figure(figure, "ht_public_running_means")

,Industry,Degree,Generated baseline,Generated exposure effect,Probability of exposure zero,Probability of exposure one
0,Agric,2,0.81281,0.6,0.5625,0.4375
1,Food,3,1.434119,0.7,0.421875,0.578125
2,Soda,2,1.303585,0.6,0.5625,0.4375
3,Beer,4,1.600988,0.8,0.316406,0.683594
4,Smoke,2,1.122707,0.6,0.5625,0.4375
5,Toys,2,0.98064,0.6,0.5625,0.4375
6,Fun,2,1.567641,0.6,0.5625,0.4375
7,Books,3,1.381738,0.7,0.421875,0.578125
8,Hshld,3,1.231402,0.7,0.421875,0.578125
9,Clths,4,1.528676,0.8,0.316406,0.683594


,Quantity,Value
0,Assignments enumerated,"4,096"
1,Known average exposure effect,0.666667
2,Exact Horvitz-Thompson expectation,0.666667
3,Exact expectation error,-1.11022e-16
4,Naive expectation when both groups appear,0.755826
5,Probability both exposure groups appear,0.949848


,Estimator,Mean,Standard deviation,Error from truth,Valid repetitions
0,Horvitz-Thompson,0.675536,1.522888,0.008869,"5,000"
1,Naive observed difference,0.757438,0.162506,0.090772,"4,756"


,Estimator,5th percentile,Median,95th percentile,Minimum,Maximum
0,Horvitz-Thompson contrast,-1.731188,0.83356,3.099168,-2.96718,3.813519
1,Naive observed difference,0.468904,0.752166,1.013591,0.33258,1.217874


,Industry,Exposure,Observed outcome,Unit contribution
0,Agric,1,1.41281,3.229279
1,Food,0,1.434119,-3.399393
2,Soda,0,1.303585,-2.317484
3,Beer,1,2.400988,3.512303
4,Smoke,0,1.122707,-1.995924
5,Toys,1,1.58064,3.612892
6,Fun,1,2.167641,4.954608
7,Books,1,2.081738,3.600844
8,Hshld,1,1.931402,3.340804
9,Clths,1,2.328676,3.40652


,Check,Rejected,Explanation
0,Exposure label outside the supplied columns,Pass,Stopped as required
1,Contrasted exposures are identical,Pass,Stopped as required
2,A contrasted exposure has zero probability,Pass,Stopped as required
3,Probability row exceeds one,Pass,Stopped as required


<Figure size 880x520 with 1 Axes>

<Figure size 880x520 with 1 Axes>

<Figure size 880x520 with 1 Axes>